# IPL Data Visualization Dashboard (Task 3)\n
This notebook uses the IPL ball-by-ball dataset (2008–2022) and creates bar, line, pie, histogram, and scatter plots using Pandas, Matplotlib, and Seaborn.

## Step 1: Upload the CSV file\n
In Google Colab, upload `IPL_Ball_by_Ball_2008_2022.csv` when the upload window appears.

In [ ]:
from google.colab import files\n
uploaded = files.upload()\n
\n
import io\n
import pandas as pd\n
file_name = next(iter(uploaded))\n
df = pd.read_csv(io.BytesIO(uploaded[file_name]))\n
print('Dataset loaded successfully!')\n
print('Rows and columns:', df.shape)\n
display(df.head())

## Step 2: Inspect and prepare the data\n
Check column names, data types, missing values, and duplicates. Missing text values are kept as missing because they can mean that no extra or wicket event occurred.

In [ ]:
print('Column names:\\n', df.columns.tolist())\n
print('\\nData types:\\n')\n
print(df.dtypes)\n
print('\\nMissing values:\\n')\n
print(df.isnull().sum())\n
print('\\nDuplicate rows:', df.duplicated().sum())\n
\n
# Remove exact duplicate rows and standardize column text\n
df = df.drop_duplicates().copy()\n
df['BattingTeam'] = df['BattingTeam'].astype('string').str.strip()\n
for col in ['batter', 'bowler', 'kind', 'extra_type']:\n
    if col in df.columns:\n
        df[col] = df[col].astype('string').str.strip()\n
\n
# Convert numeric columns safely\n
numeric_cols = ['ID','innings','overs','ballnumber','batsman_run','extras_run','total_run','isWicketDelivery']\n
for col in numeric_cols:\n
    df[col] = pd.to_numeric(df[col], errors='coerce')\n
\n
print('Data prepared. Shape:', df.shape)

## Step 3: Create summary tables and dashboard charts

In [ ]:
import matplotlib.pyplot as plt\n
import seaborn as sns\n
sns.set_theme(style='whitegrid')\n
\n
# Useful summary tables\n
team_runs = df.groupby('BattingTeam', dropna=True)['total_run'].sum().sort_values(ascending=False)\n
match_totals = df.groupby('ID')['total_run'].sum()\n
over_runs = df.groupby('overs')['total_run'].mean().sort_index()\n
wicket_kinds = df.loc[df['isWicketDelivery'] == 1, 'kind'].dropna().value_counts()\n
batter_summary = df.groupby('batter').agg(runs=('batsman_run','sum'), balls=('batsman_run','count'))\n
batter_summary = batter_summary[batter_summary['balls'] >= 100].sort_values('runs', ascending=False)\n
\n
print('Top 10 teams by total runs:')\n
display(team_runs.head(10).rename('Total Runs').to_frame())\n
print('Top 10 batters (minimum 100 balls faced):')\n
display(batter_summary.head(10))\n
print('Total matches in dataset:', df['ID'].nunique())\n
print('Total runs:', int(df['total_run'].sum()))\n
print('Total wickets:', int(df['isWicketDelivery'].sum()))

In [ ]:
# 1. BAR CHART: Top 10 teams by total runs\n
plt.figure(figsize=(12,6))\n
team_runs.head(10).sort_values().plot(kind='barh', color='steelblue')\n
plt.title('Top 10 IPL Teams by Total Runs (2008–2022)')\n
plt.xlabel('Total Runs')\n
plt.ylabel('Batting Team')\n
plt.tight_layout()\n
plt.show()

In [ ]:
# 2. LINE CHART: Average runs scored in each over\n
plt.figure(figsize=(12,6))\n
over_runs.plot(marker='o', linewidth=2, color='darkorange')\n
plt.title('Average Runs Scored per Over')\n
plt.xlabel('Over Number (0 = first over)')\n
plt.ylabel('Average Runs per Ball')\n
plt.xticks(range(int(df['overs'].max()) + 1))\n
plt.tight_layout()\n
plt.show()

In [ ]:
# 3. PIE CHART: Wicket types\n
if not wicket_kinds.empty:\n
    plt.figure(figsize=(8,8))\n
    wicket_kinds.plot(kind='pie', autopct='%1.1f%%', startangle=90)\n
    plt.title('Distribution of Wicket Types')\n
    plt.ylabel('')\n
    plt.tight_layout()\n
    plt.show()\n
else:\n
    print('No wicket type data available.')

In [ ]:
# 4. HISTOGRAM: Runs scored from the bat per delivery\n
plt.figure(figsize=(10,6))\n
sns.histplot(df['batsman_run'].dropna(), bins=8, kde=False)\n
plt.title('Distribution of Batter Runs per Ball')\n
plt.xlabel('Runs scored by batter on a ball')\n
plt.ylabel('Number of deliveries')\n
plt.tight_layout()\n
plt.show()

In [ ]:
# 5. SCATTER PLOT: Balls faced vs runs scored by top batters\n
top_batters = batter_summary.head(20).reset_index()\n
plt.figure(figsize=(10,6))\n
sns.scatterplot(data=top_batters, x='balls', y='runs', hue='batter', s=100)\n
plt.title('Runs vs Balls Faced – Top 20 Batters (100+ balls)')\n
plt.xlabel('Balls faced (deliveries recorded)')\n
plt.ylabel('Total runs scored')\n
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')\n
plt.tight_layout()\n
plt.show()

## Step 4: Display simple insights\n
The printed results below help explain what the charts show.

In [ ]:
if not team_runs.empty:\n
    print('Highest-scoring team in this dataset:', team_runs.index[0], '-', int(team_runs.iloc[0]), 'runs')\n
if not batter_summary.empty:\n
    print('Top batter by total runs (minimum 100 balls):', batter_summary.index[0], '-', int(batter_summary.iloc[0]['runs']), 'runs')\n
if not wicket_kinds.empty:\n
    print('Most common wicket type:', wicket_kinds.index[0], '-', int(wicket_kinds.iloc[0]), 'deliveries')\n
print('The line chart compares average runs by over number; the histogram shows how often each batter-run value occurs.')

## Step 5: Save the prepared dataset as a CSV\n
This saves the data used for the analysis.

In [ ]:
output_file = 'IPL_Ball_by_Ball_2008_2022_prepared.csv'\n
df.to_csv(output_file, index=False)\n
print('Saved:', output_file)\n
files.download(output_file)